<table style="width:100%">
  <tr>
    <td valign="top"><img src="../data/img/FER_logo_2.png" width=300 height=80 align="left"></td>
    <td valign="top"><img src="../data/img/LARES_2_transparent.png" width=250 height=80 align="right"></td>
  </tr>
 </table>

# House Prices - Advanced Regression Techniques
Predict sales prices and practice the basic concepts of machine learning...
Our competition with dataset, competition rules and other information can be found on [Kaggle competition](https://www.kaggle.com/t/2d8eb2d88f9090f65257fb7975163ae5)

The original dataset and many exemplary notebooks can be found on [Kaggle original dataset](https://www.kaggle.com/c/house-prices-advanced-regression-techniques).

![title](../data/img/house_prices.jpg)

## Competition

1. Create your own [Kaggle profile](https://www.kaggle.com/account/login?phase=startRegisterTab&returnUrl=%2F),
2. Join our "AI Bootcamp house prices" competition using the
[invitation link](https://www.kaggle.com/t/2d8eb2d88f9090f65257fb7975163ae5)
3. Analyse the [dataset](https://www.kaggle.com/competitions/ai-bootcamp-house-prices/data) and [competition rules](https://www.kaggle.com/competitions/ai-bootcamp-house-prices/rules),
4. Take special care with [metrics](https://www.kaggle.com/c/ai-bootcamp-house-prices/overview/evaluation),
5. Check out publicly shared [notebooks](https://www.kaggle.com/c/house-prices-advanced-regression-techniques/code?competitionId=5407&sortBy=voteCount)  with solutions using the same dataset (public version),
6. Tune models and generate predictions,
7. Generate a [submission.csv file](https://www.kaggle.com/c/ai-bootcamp-house-prices/data?select=sample_submission.csv) and submit it for grading using the Submit Predictions button in the upper-right corner of the competition page.

Have fun and good luck!

## Import libraries

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

## Load datasets

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
import lightgbm as lgb

In [ ]:
# import training data
df = pd.read_csv('../data/housing_prices/Kaggle/train.csv', index_col='Id')
# remove features with missing data
total = df.isnull().sum().sort_values(ascending=False)
percent = (df.isnull().sum() / df.isnull().count() * 100).sort_values(ascending=False)
missing_data = pd.concat([total, percent], axis=1, keys=['Total', 'Percent'])
df = df.drop((missing_data[missing_data['Percent'] > 0.5]).index, axis=1)
df = df.drop(df.loc[df['Electrical'].isnull()].index)
# use only the most correlated features
corr = df.corr(numeric_only=True)
relevant_inp = np.abs(corr['SalePrice']).sort_values(ascending=False).index[0:11]
# separate X and y and split to train and validation sets
X = df.loc[df.index, relevant_inp[1:]].apply(np.log1p)  # logarithm transformation
y = np.log1p(df.loc[df.index, relevant_inp[0]])  # logarithm transformation
# separate train and validation datasets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=1)

In [ ]:
# Train a simple LightGBM model
model = lgb.LGBMRegressor(random_state=42, n_estimators=100, verbose=-1)
model.fit(X_train, y_train)

# TODO: Can you improve this model?
# Try different hyperparameters or use GridSearchCV.

In [ ]:
# Generate predictions and calculate RMSLE
y_pred = model.predict(X_val)

# Convert back from log scale for evaluation
y_val_original = np.expm1(y_val)
y_pred_original = np.expm1(y_pred)

# Calculate RMSLE
rmsle = np.sqrt(mean_squared_log_error(y_val_original, y_pred_original))
print('RMSLE: %.4f' % rmsle)

In [ ]:
# Train final model on all data
model_final = lgb.LGBMRegressor(random_state=42, n_estimators=100, verbose=-1)
model_final.fit(X, y)

In [ ]:
# Import test dataset
X_test = pd.read_csv('../data/housing_prices/Kaggle/test.csv', index_col='Id')
X_test = X_test.loc[:,relevant_inp[1:]].apply(np.log1p)  # logarithm transformation
X_test

In [ ]:
# Generate predictions on test data
y_pred_log = model_final.predict(X_test)
y_pred = np.expm1(y_pred_log)  # Convert back from log scale

## Create submission csv file
Take y_pred object (array of floats) and create a submission CSV file for uploading to Kaggle.

In [ ]:
df_pred = pd.DataFrame(y_pred)
s_sub = pd.read_csv("../data/housing_prices/Kaggle/sample_submission.csv")
sub_df = pd.concat([s_sub['Id'], df_pred], axis=1)
sub_df.columns=['Id', 'SalePrice']
sub_df.to_csv("../data/housing_prices/Kaggle/my_submission.csv", index=False)

In [ ]:
pd.read_csv("../data/housing_prices/Kaggle/my_submission.csv")


_Course: AI Bootcamp: Foundations of AI_  
_Notebook: 8_Kaggle_  
_Kaggle Competition: [AI Bootcamp House Prices](https://www.kaggle.com/competitions/ai-bootcamp-house-prices)_

_University of Zagreb Faculty of Electrical Engineering and Computing_  
_Laboratory for Renewable Energy Systems_  
_Website: [www.lares.fer.hr](https://www.lares.fer.hr/)_  
_Contact: [Hrvoje Novak](mailto:hrvoje.novak@fer.hr)_


